In [1]:
# ============================================================
# STARTUP (Target-Agnostic)
# ============================================================
# Install required packages, import common libraries,
# mount Google Drive, configure project paths, and
# create common project directories.

!pip -q install "datasets<4.0.0" "huggingface_hub<0.25"

import io
import csv
import json
import time
import hashlib
import sys
import os
import importlib

from pathlib import Path
from typing import List, Dict, Optional, Tuple

from google.colab import drive
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/My Drive/DIP_Project')
SRC_DIR = PROJECT_ROOT / 'src'

if not SRC_DIR.exists():
    raise FileNotFoundError(f'Could not find src directory: {SRC_DIR}')

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

COMMON_DIRS = [
    'data/raw',
    'data/processed',
    'data/metadata',
    'outputs/images',
    'outputs/logs',
]

for d in COMMON_DIRS:
    os.makedirs(PROJECT_ROOT / d, exist_ok=True)

print('Startup complete.')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_DIR:', SRC_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Startup complete.
PROJECT_ROOT: /content/drive/My Drive/DIP_Project
SRC_DIR: /content/drive/My Drive/DIP_Project/src


In [2]:
# ============================================================
# CELL 0: NOTEBOOK SUMMARY
# DATASET BUILDER DRIVER NOTEBOOK
# ============================================================

# Purpose:
#   - Build standardized image datasets from multiple sources
#   - Execute one dataset target at a time (DiffusionDB, SDXL, ImageNet, COCO)
#   - Maintain a consistent, reusable pipeline across all sources

# Design Overview:
#   - This notebook implements a modular dataset builder pipeline
#   - Common workflow logic is centralized here
#   - Dataset-specific logic is encapsulated in target modules

# Core Responsibilities (This Notebook):
#   - target selection and dynamic module loading
#   - directory and environment setup
#   - CSV metadata creation and management
#   - image validation (size, format)
#   - hashing and duplicate detection
#   - deterministic filename generation
#   - image saving and metadata recording
#   - final dataset verification and inspection

# Target Module Responsibilities (src/targets/*.py):
#   - load_source_dataset(...)
#   - get_candidate_records(...)
#   - Provide dataset-specific access and record extraction only

# Standard Candidate Record Format:
#   Each target module must return records with:
#     - source_id   : unique identifier from source dataset
#     - source_ref  : reference string (e.g., Hugging Face path)
#     - image_obj   : PIL Image object

# Key Pipeline Features:
#   - Deterministic naming:
#       Filenames are based on accepted index (not filesystem state)
#       → ensures reproducibility and rebuild capability
#
#   - Idempotent execution:
#       Safe to rerun without duplicating work
#
#   - Resume support:
#       Existing images are reused (no re-download)
#
#   - Metadata rebuild capability:
#       CSV can be regenerated from existing images without rebuilding dataset
#
#   - Data integrity guarantees:
#       Image save + CSV write are treated as a single atomic operation

# Valid TARGET_NAME values:
#   - diffusiondb
#   - sdxl
#   - imagenet
#   - coco

# Notes:
#   - This notebook is target-agnostic except for TARGET_NAME (Cell 1)
#   - Large datasets are stored in Google Drive, not GitHub
#   - DiffusionDB requires:
#         trust_remote_code=True
#     in load_dataset(...) inside the target module to avoid prompts

# ============================================================
# Expected Google Drive Layout
# ============================================================
# My Drive/
# └── DIP_Project/
#     ├── src/
#     │   ├── targets/
#     │   │   ├── diffusiondb_target.py
#     │   │   ├── sdxl_target.py
#     │   │   ├── imagenet_target.py
#     │   │   └── coco_target.py
#     │   └── utils/
#     │
#     ├── notebooks/
#     │   └── 01_Dataset_Builder.ipynb
#     │
#     ├── data/
#     │   ├── raw/         # optional staging
#     │   ├── processed/   # saved images
#     │   └── metadata/    # CSV + hash files
#     │
#     └── outputs/
#         ├── images/      # optional alternative output path
#         └── logs/        # logs, debug output


In [3]:
# ============================================================
# Cell 1: Select Target Dataset
# ============================================================

TARGETS = {
    'diffusiondb': {
        'display_name': 'DiffusionDB',
        'module_name': 'targets.diffusiondb_target',
        'source_dataset': 'DiffusionDB',
        'filename_label': 'ai',
        'dataset_code': 'diff',
    },
    'sdxl': {
        'display_name': 'SDXL Generated (10K)',
        'module_name': 'targets.sdxl_target',
        'source_dataset': 'SDXL_Generated_10K',
        'filename_label': 'ai',
        'dataset_code': 'sdxl',
    },
    'imagenet': {
        'display_name': 'ImageNet (256)',
        'module_name': 'targets.imagenet_target',
        'source_dataset': 'ImageNet_1K_256',
        'filename_label': 'rl',
        'dataset_code': 'imgn',
    },
    'coco': {
        'display_name': 'MS COCO 2017',
        'module_name': 'targets.coco_target',
        'source_dataset': 'MS_COCO_2017',
        'filename_label': 'rl',
        'dataset_code': 'coco',
    },
}

# Change only this line to switch datasets.
TARGET_NAME = 'diffusiondb'

if TARGET_NAME not in TARGETS:
    raise ValueError(f'Unknown TARGET_NAME: {TARGET_NAME}')

TARGET_CONFIG = TARGETS[TARGET_NAME]
TARGET_DISPLAY_NAME = TARGET_CONFIG['display_name']
SOURCE_DATASET = TARGET_CONFIG['source_dataset']
FILENAME_LABEL = TARGET_CONFIG['filename_label']
DATASET_CODE = TARGET_CONFIG['dataset_code']

print(f'Target selected: {TARGET_NAME}')
print(f'Display name   : {TARGET_DISPLAY_NAME}')
print(f'Source dataset : {SOURCE_DATASET}')
print(f'Filename label : {FILENAME_LABEL}')
print(f'Dataset code   : {DATASET_CODE}')
print(f'Module         : {TARGET_CONFIG["module_name"]}')


Target selected: diffusiondb
Display name   : DiffusionDB
Source dataset : DiffusionDB
Filename label : ai
Dataset code   : diff
Module         : targets.diffusiondb_target


In [4]:
# ============================================================
# Cell 2: Load Target Module
# ============================================================

target_module = importlib.import_module(TARGET_CONFIG['module_name'])
importlib.reload(target_module)

load_source_dataset = target_module.load_source_dataset
get_candidate_records = target_module.get_candidate_records

print(f'Imported target module for {TARGET_DISPLAY_NAME}: OK')
print('Module file:', target_module.__file__)


Imported target module for DiffusionDB: OK
Module file: /content/drive/My Drive/DIP_Project/src/targets/diffusiondb_target.py


In [5]:
# ============================================================
# Cell 3: Common Configuration
# ============================================================

TARGET_ACCEPTED = 3000
MIN_WIDTH = 256
MIN_HEIGHT = 256
BATCH_ID_START = 1
BATCH_SIZE = 200
RANDOM_SEED = 42
SLEEP_BETWEEN_BATCHES = 1.0

BASE_DIR = PROJECT_ROOT / 'data'
IMAGES_DIR = BASE_DIR / 'raw' / SOURCE_DATASET / 'images'
METADATA_DIR = BASE_DIR / 'metadata'
CSV_PATH = METADATA_DIR / f'{SOURCE_DATASET.lower()}_metadata.csv'
SOURCE_HASH_PATH = METADATA_DIR / f'{SOURCE_DATASET.lower()}_source_hashes.json'
GLOBAL_HASH_PATH = METADATA_DIR / 'global_hashes.json'

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print('IMAGES_DIR      :', IMAGES_DIR)
print('METADATA_DIR    :', METADATA_DIR)
print('CSV_PATH        :', CSV_PATH)
print('SOURCE_HASH_PATH:', SOURCE_HASH_PATH)
print('GLOBAL_HASH_PATH:', GLOBAL_HASH_PATH)


IMAGES_DIR      : /content/drive/My Drive/DIP_Project/data/raw/DiffusionDB/images
METADATA_DIR    : /content/drive/My Drive/DIP_Project/data/metadata
CSV_PATH        : /content/drive/My Drive/DIP_Project/data/metadata/diffusiondb_metadata.csv
SOURCE_HASH_PATH: /content/drive/My Drive/DIP_Project/data/metadata/diffusiondb_source_hashes.json
GLOBAL_HASH_PATH: /content/drive/My Drive/DIP_Project/data/metadata/global_hashes.json


In [6]:
# ============================================================
# Cell 4: CSV Setup
# ============================================================

CSV_COLUMNS = [
    'filename',
    'label',
    'dataset_code',
    'source_name',
    'source_id',
    'source_ref',
    'original_width',
    'original_height',
    'saved_path',
    'sha256',
    'batch_id',
]

def initialize_csv(csv_path: Path, columns: List[str]) -> None:
    if not csv_path.exists():
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            writer.writeheader()

initialize_csv(CSV_PATH, CSV_COLUMNS)
print('CSV initialized:', CSV_PATH)


CSV initialized: /content/drive/My Drive/DIP_Project/data/metadata/diffusiondb_metadata.csv


In [7]:
# ============================================================
# Cell 5: Hash Utilities
# ============================================================

def load_hash_set(hash_path: Path) -> set:
    if hash_path.exists():
        with open(hash_path, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()

def save_hash_set(hash_path: Path, hash_set: set) -> None:
    with open(hash_path, 'w', encoding='utf-8') as f:
        json.dump(sorted(list(hash_set)), f, indent=2)

source_hashes = load_hash_set(SOURCE_HASH_PATH)
global_hashes = load_hash_set(GLOBAL_HASH_PATH)

print('Loaded source hashes:', len(source_hashes))
print('Loaded global hashes:', len(global_hashes))


Loaded source hashes: 0
Loaded global hashes: 0


In [8]:
# ============================================================
# Cell 6: Filename and Counting Utilities
# ============================================================

def count_existing_images(images_dir: Path) -> int:
    return len(list(images_dir.glob('*.png')))

def next_index(images_dir: Path) -> int:
    return count_existing_images(images_dir) + 1

def build_filename(label: str, dataset_code: str, idx: int) -> str:
    return f'{label}_{dataset_code}_{idx:06d}.png'


In [9]:
# ============================================================
# Cell 7: Image Utilities
# ============================================================

def normalize_image(img: Image.Image) -> Image.Image:
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return img

def image_meets_size_requirement(
    img: Image.Image,
    min_width: int = MIN_WIDTH,
    min_height: int = MIN_HEIGHT,
) -> bool:
    w, h = img.size
    return (w >= min_width) and (h >= min_height)

def compute_sha256_for_normalized_png(img: Image.Image) -> str:
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    return hashlib.sha256(buffer.getvalue()).hexdigest()


In [10]:
# ============================================================
# Cell 8: CSV Append Utility
# ============================================================

def append_csv_row(csv_path: Path, row: Dict) -> None:
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writerow(row)


In [11]:
# ============================================================
# Cell 9: Load Source Dataset
# ============================================================

source_ds = load_source_dataset(seed=RANDOM_SEED)
print(f'{TARGET_DISPLAY_NAME} source dataset loaded.')
print('Type:', type(source_ds))
print('Current accepted images:', count_existing_images(IMAGES_DIR))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DiffusionDB source dataset loaded.
Type: <class 'datasets.arrow_dataset.Dataset'>
Current accepted images: 3000


In [12]:
# ============================================================
# Cell 10: Preview Candidate Records
# ============================================================

candidate_records = get_candidate_records(
    source_ds,
    batch_size=BATCH_SIZE,
    batch_id=BATCH_ID_START,
)

print(f'Candidate records collected: {len(candidate_records)}')

if len(candidate_records) > 0:
    print('First candidate record:')
    print(candidate_records[0])


Candidate records collected: 200
First candidate record:
{'source_id': 'diffusiondb_200', 'source_ref': 'huggingface://poloclub/diffusiondb/2m_random_5k/200', 'image_obj': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=704x512 at 0x79AB8D9DB680>}


In [13]:
# ============================================================
# Cell 11: Process One Candidate Record
# ============================================================

def process_candidate(record: Dict, batch_id: int, idx: int) -> Tuple[bool, Optional[str]]:
    global source_hashes, global_hashes

    source_id = record.get('source_id', '')
    source_ref = record.get('source_ref', '')
    img = record.get('image_obj', None)

    if img is None:
        return False, 'missing_image'

    try:
        img = normalize_image(img)
    except Exception:
        return False, 'invalid_image'

    width, height = img.size

    if not image_meets_size_requirement(img):
        return False, 'too_small'

    try:
        sha256 = compute_sha256_for_normalized_png(img)
    except Exception:
        return False, 'hash_failed'

    if sha256 in source_hashes:
        return False, 'duplicate_within_source'

    if sha256 in global_hashes:
        return False, 'duplicate_global'

    filename = build_filename(FILENAME_LABEL, DATASET_CODE, idx)
    save_path = IMAGES_DIR / filename

    row = {
        'filename': filename,
        'label': FILENAME_LABEL,
        'dataset_code': DATASET_CODE,
        'source_name': SOURCE_DATASET,
        'source_id': source_id,
        'source_ref': source_ref,
        'original_width': width,
        'original_height': height,
        'saved_path': str(save_path),
        'sha256': sha256,
        'batch_id': batch_id,
    }

    try:
        image_already_exists = save_path.exists()

        if not image_already_exists:
            img.save(save_path, format='PNG')

        append_csv_row(CSV_PATH, row)

    except Exception:
        # If we created the image but CSV failed, clean it up
        try:
            if not image_already_exists and save_path.exists():
                save_path.unlink()
        except Exception:
            pass

        return False, 'save_or_csv_failed'

    source_hashes.add(sha256)
    global_hashes.add(sha256)

    return True, None


In [14]:
# ============================================================
# Cell 12: Build Dataset for Selected Source
# ============================================================

def build_dataset_for_source(
    batch_size: int = BATCH_SIZE,
    sleep_between_batches: float = SLEEP_BETWEEN_BATCHES,
) -> None:
    batch_id = BATCH_ID_START
    accepted_total = 0

    print(f'Starting source: {TARGET_NAME}')
    print(f'Current accepted images: {accepted_total}')
    print(f'Target accepted images: {TARGET_ACCEPTED}')
    print('-' * 60)

    while accepted_total < TARGET_ACCEPTED:
        needed = TARGET_ACCEPTED - accepted_total
        print(f'\nBatch {batch_id} | accepted so far: {accepted_total} | still needed: {needed}')

        records = get_candidate_records(
            source_ds,
            batch_size=batch_size,
            batch_id=batch_id,
        )

        if not records:
            print('No more records returned by target module. Stopping.')
            break

        accepted_in_batch = 0
        rejected_counts = {}

        for record in tqdm(records, desc=f'Batch {batch_id}'):
            next_idx = accepted_total + 1
            accepted, reason = process_candidate(record, batch_id, next_idx)

            if accepted:
                accepted_in_batch += 1
                accepted_total += 1

                if accepted_total >= TARGET_ACCEPTED:
                    break
            else:
                rejected_counts[reason] = rejected_counts.get(reason, 0) + 1

        save_hash_set(SOURCE_HASH_PATH, source_hashes)
        save_hash_set(GLOBAL_HASH_PATH, global_hashes)

        print(f'Accepted this batch: {accepted_in_batch}')
        if rejected_counts:
            print('Rejected counts:', rejected_counts)

        batch_id += 1
        if accepted_total < TARGET_ACCEPTED:
            time.sleep(sleep_between_batches)

    print('\nBuild complete.')
    print(f'Final accepted images: {accepted_total}')


In [15]:
# ============================================================
# Cell 13: Verify Saved Output
# ============================================================

def csv_row_count(csv_path: Path) -> int:
    if not csv_path.exists():
        return 0
    with open(csv_path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)

def verify_source_output() -> None:
    image_count = count_existing_images(IMAGES_DIR)
    row_count = csv_row_count(CSV_PATH)

    print(f'Source: {SOURCE_DATASET}')
    print(f'Image count: {image_count}')
    print(f'CSV row count: {row_count}')

    if image_count == row_count:
        print('PASS: image count matches CSV row count')
    else:
        print('FAIL: image count does not match CSV row count')


In [16]:
# ============================================================
# Cell 14: Run Builder
# ============================================================

build_dataset_for_source(batch_size=BATCH_SIZE)


Starting source: diffusiondb
Current accepted images: 0
Target accepted images: 3000
------------------------------------------------------------

Batch 1 | accepted so far: 0 | still needed: 3000


Batch 1:   0%|          | 0/200 [00:00<?, ?it/s]

Accepted this batch: 200

Batch 2 | accepted so far: 200 | still needed: 2800


Batch 2:   0%|          | 0/200 [00:00<?, ?it/s]

Accepted this batch: 200

Batch 3 | accepted so far: 400 | still needed: 2600


...

Accepted this batch: 199
Rejected counts: {'duplicate_within_source': 1}

Batch 15 | accepted so far: 2791 | still needed: 209


Batch 15:   0%|          | 0/200 [00:00<?, ?it/s]

Accepted this batch: 199
Rejected counts: {'duplicate_within_source': 1}

Batch 16 | accepted so far: 2990 | still needed: 10


Batch 16:   0%|          | 0/200 [00:00<?, ?it/s]

Accepted this batch: 10

Build complete.
Final accepted images: 3000


In [17]:
# ============================================================
# Cell 15: Final Verification
# ============================================================

verify_source_output()


Source: DiffusionDB
Image count: 3000
CSV row count: 3000
PASS: image count matches CSV row count


In [28]:
# ============================================================
# Cell 16: Final Inspection Cell
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 60)
print("FINAL DATASET INSPECTION")
print("=" * 60)

# --- Load CSV ---
try:
    df = pd.read_csv(CSV_PATH)
    print(f"CSV loaded successfully: {len(df)} rows")
except Exception as e:
    print(f"ERROR loading CSV: {e}")
    raise

# --- Image files ---
image_files = list(Path(IMAGES_DIR).glob("*.png"))
image_count = len(image_files)

print(f"Image files found: {image_count}")

# --- Basic consistency check ---
if image_count == len(df):
    print("PASS: Image count matches CSV row count")
else:
    print("FAIL: Image count does NOT match CSV row count")

# --- Filename uniqueness ---
unique_filenames = df['filename'].nunique()
if unique_filenames == len(df):
    print("PASS: All filenames are unique")
else:
    print("FAIL: Duplicate filenames detected")

# --- Missing files check ---
missing_files = []
for fname in df['filename']:
    if not (Path(IMAGES_DIR) / fname).exists():
        missing_files.append(fname)

if not missing_files:
    print("PASS: All CSV filenames exist in image directory")
else:
    print(f"FAIL: {len(missing_files)} missing image files")

# --- Sequential filename check ---
expected_filenames = [
    build_filename(FILENAME_LABEL, DATASET_CODE, i)
    for i in range(1, len(df) + 1)
]

if set(expected_filenames) == set(df['filename']):
    print("PASS: Filenames follow expected sequential pattern")
else:
    print("WARNING: Filenames may not be perfectly sequential")

# --- Sample inspection ---
print("\nSample rows:")
print(df.sample(3))

# --- Summary ---
print("\nSummary:")
print(f"Total images: {image_count}")
print(f"Total CSV rows: {len(df)}")

print("=" * 60)
print("INSPECTION COMPLETE")
print("=" * 60)

FINAL DATASET INSPECTION
CSV loaded successfully: 3000 rows
Image files found: 3000
PASS: Image count matches CSV row count
PASS: All filenames are unique
PASS: All CSV filenames exist in image directory
PASS: Filenames follow expected sequential pattern

Sample rows:
                filename label dataset_code  source_name         source_id  \
2499  ai_diff_002500.png    ai         diff  DiffusionDB  diffusiondb_2707   
1838  ai_diff_001839.png    ai         diff  DiffusionDB  diffusiondb_2044   
36    ai_diff_000037.png    ai         diff  DiffusionDB   diffusiondb_236   

                                             source_ref  original_width  \
2499  huggingface://poloclub/diffusiondb/2m_random_5...             512   
1838  huggingface://poloclub/diffusiondb/2m_random_5...             512   
36    huggingface://poloclub/diffusiondb/2m_random_5...             512   

      original_height                                         saved_path  \
2499              512  /content/drive/My 